In [ ]:
import kagglehub
path = kagglehub.dataset_download("columbine/imdb-dataset-sentiment-analysis-in-csv-format")

100%|██████████| 25.7M/25.7M [00:02<00:00, 11.2MB/s]

Extracting files...


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dftrain=pd.read_csv("/root/.cache/kagglehub/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format/versions/1/Train.csv")
dftest=pd.read_csv("/root/.cache/kagglehub/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format/versions/1/Test.csv")
dfvalid=pd.read_csv("/root/.cache/kagglehub/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format/versions/1/Valid.csv")

In [ ]:
dftrain.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [ ]:
print(dftrain.isnull().sum())
print(dftest.isnull().sum())
print(dfvalid.isnull().sum())

text     0
label    0
dtype: int64
text     0
label    0
dtype: int64
text     0
label    0
dtype: int64


In [ ]:
print(dftrain.duplicated().sum())
print(dftest.duplicated().sum())
print(dfvalid.duplicated().sum())


277
5
2


In [ ]:
dftrain=dftrain.drop_duplicates()
dftest=dftest.drop_duplicates()
dfvalid=dfvalid.drop_duplicates()

In [ ]:
import string

vocab = set()

for i in dftrain['text']:

    i = i.lower()
    i = i.translate(str.maketrans('', '', string.punctuation))

    words = i.split()

    for word in words:
        vocab.add(word)

dictIndex = {}
index = 0

for word in vocab:
    dictIndex[word] = index
    index += 1

In [ ]:
def encoding(text):
  text=text.lower()
  text=text.translate(str.maketrans('','',string.punctuation))
  texts=text.split()
  encoding=[]
  for i in texts:
    if i in dictIndex:
      encoding.append(dictIndex[i])
  return encoding
dftrain['encoded_text']=dftrain['text'].apply(encoding)


In [ ]:
dftrain.head()

,text,label,encoded_text
0,I grew up (b. 1965) watching and loving the Th...,0,"[14186, 117379, 144503, 101614, 60615, 68146, ..."
1,"When I put this movie in my DVD player, and sa...",0,"[122614, 14186, 29616, 154272, 14681, 88882, 7..."
2,Why do people who do not know what a particula...,0,"[47203, 156817, 64592, 135655, 156817, 120942,..."
3,Even though I have great interest in Biblical ...,0,"[118738, 149992, 14186, 105594, 2557, 35482, 8..."
4,Im a die hard Dads Army fan and nothing will e...,1,"[119357, 86499, 68927, 9429, 128711, 76357, 61..."


In [ ]:
maxlen = 300

def paddingadd(sequence, maxlen):
  if len(sequence) > maxlen:
     sequence = sequence[:maxlen]
  else:
     sequence = sequence + [0] * (maxlen - len(sequence))
  return sequence

def padding(sequence, maxlen):
  return paddingadd(sequence, maxlen)

dftrain['padded_text'] = dftrain['encoded_text'].apply(lambda x: padding(x, maxlen))

In [ ]:
import torch

x=torch.tensor(dftrain['padded_text'].tolist())
y=torch.tensor(dftrain['label'].values)


In [ ]:
from torch.utils.data import DataLoader,Dataset
class Data(Dataset):
  def __init__(self,x,y):
    self.x=x
    self.y=y

  def __len__(self):

    return len(self.x)

  def __getitem__(self,index):
    return self.x[index],self.y[index]

In [ ]:
trainset=Data(x,y)
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)


In [ ]:
import torch.nn as nn
class RNN(nn.Module):
  def __init__(self,vocabsize,hidden_dim,embed_dim,output_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocabsize,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
    self.dropout = nn.Dropout(0.3)                         # stops cheating
    self.fc = nn.Linear(hidden_dim, output_dim)
  def forward(self, x):
    x = self.embedding(x)                  # (batch, seq_len) -> (batch, seq_len, embed_dim)
    out, (hidden, cell) = self.lstm(x)     # read the whole sentence
    last = hidden[-1]                      # final note in the notebook
    last = self.dropout(last)
    return self.fc(last)

In [ ]:
vocab_size = len(dictIndex) + 1

model = RNN(
    vocabsize=vocab_size,
    embed_dim=128,
    hidden_dim=64,
    output_dim=2
)

In [ ]:
import torch.optim as optim
criterion = nn.BCEWithLogitsLoss()               # for yes/no
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Switch to CrossEntropyLoss since the model output_dim is 2
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in trainloader:
        # Forward pass
        output = model(batch_X) # shape: [batch_size, 2]

        # Calculate loss (CrossEntropyLoss expects target of shape [batch_size] with class indices)
        loss = criterion(output, batch_y.long())

        # Remove old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(trainloader)
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {average_loss:.4f}")

Epoch 1/5, Loss: 0.6943
Epoch 2/5, Loss: 0.6812
Epoch 3/5, Loss: 0.5892
Epoch 4/5, Loss: 0.4203
Epoch 5/5, Loss: 0.2796


In [ ]:
dftest['encodded_text']=dftest['text'].apply(encoding)
dftest['padded_text']=dftest['encodded_text'].apply(lambda x: padding(x,maxlen))


In [ ]:
xtest=torch.tensor(dftest['padded_text'].tolist())
ytest=torch.tensor(dftest['label'].values)

In [ ]:
# Ensure xtest and ytest are defined from dftest
xtest = torch.tensor(dftest['padded_text'].tolist())
ytest = torch.tensor(dftest['label'].values)

test_dataset = Data(
    xtest,
    ytest
)

# =========================================================
# 14. TEST DATALOADER
# =========================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

# =========================================================
# 15. TEST ACCURACY
# =========================================================

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        # Model prediction (shape: [batch_size, 2])
        output = model(batch_X)

        # Get the predicted class index with the highest logit
        predictions = torch.argmax(output, dim=1)

        # Count correct predictions
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 83.72%
